# 05 — Experimental Protocol

## Projet : AI Business Advisor V2

### Objectif

Cette étape définit le protocole expérimental avant l'entraînement
des modèles.

Les objectifs sont :

- définir les jeux Train / Validation / Test ;
- empêcher toute fuite d'information vers le jeu de test ;
- définir les stratégies de validation ;
- définir le preprocessing ;
- définir les métriques adaptées à chaque problème ;
- garantir la reproductibilité des expériences ;
- sauvegarder les partitions afin que tous les modèles soient
  comparés sur exactement les mêmes observations.

Trois problèmes sont étudiés :

1. Claim Occurrence — classification binaire ;
2. Claim Frequency — modélisation d'un nombre de sinistres ;
3. Fraud Detection — classification binaire avec dimension temporelle.

Le jeu de test final ne sera utilisé qu'après sélection définitive
du modèle et de ses hyperparamètres.

In [1]:
import pandas as pd
import numpy as np

import json

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

RANDOM_STATE = 42

In [2]:
cwd = Path.cwd()

possible_processed_paths = [
    cwd / "data" / "processed",
    cwd / "data-science" / "data" / "processed",
    cwd.parent / "data" / "processed"
]

PROCESSED_DIR = None

for path in possible_processed_paths:
    if path.exists():
        PROCESSED_DIR = path
        break

if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "Impossible de localiser data/processed"
    )

DATA_DIR = PROCESSED_DIR.parent
METADATA_DIR = DATA_DIR / "metadata"

REPORTS_DIR = (
    PROCESSED_DIR.parent.parent
    / "reports"
    / "experimental_protocol"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROCESSED :", PROCESSED_DIR.resolve())
print("METADATA  :", METADATA_DIR.resolve())
print("REPORTS   :", REPORTS_DIR.resolve())

PROCESSED : C:\Users\user\ai-business-advisor-v2\data-science\data\processed
METADATA  : C:\Users\user\ai-business-advisor-v2\data-science\data\metadata
REPORTS   : C:\Users\user\ai-business-advisor-v2\data-science\reports\experimental_protocol


In [3]:
occurrence = pd.read_csv(
    PROCESSED_DIR
    / "claim_occurrence_modeling_base.csv"
)

frequency = pd.read_csv(
    PROCESSED_DIR
    / "claim_frequency_modeling_base.csv"
)

fraud = pd.read_csv(
    PROCESSED_DIR
    / "fraud_modeling_base.csv"
)

print(
    "Occurrence :",
    occurrence.shape
)

print(
    "Frequency :",
    frequency.shape
)

print(
    "Fraud :",
    fraud.shape
)

Occurrence : (676789, 12)
Frequency : (676789, 12)
Fraud : (15420, 25)


In [4]:
MANIFEST_PATH = (
    METADATA_DIR
    / "feature_manifest.json"
)

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    feature_manifest = json.load(f)

feature_manifest

{'claim_occurrence': {'target': 'has_claim',
  'numeric_features': ['Exposure',
   'VehPower',
   'VehAge',
   'DrivAge',
   'BonusMalus',
   'log_density'],
  'categorical_features': ['Area', 'VehBrand', 'VehGas', 'Region'],
  'excluded_features': ['IDpol', 'ClaimNb']},
 'claim_frequency': {'target': 'ClaimNb',
  'exposure': 'Exposure',
  'numeric_features': ['VehPower',
   'VehAge',
   'DrivAge',
   'BonusMalus',
   'log_density'],
  'categorical_features': ['Area', 'VehBrand', 'VehGas', 'Region'],
  'excluded_features': ['IDpol']},
 'fraud_detection': {'target': 'FraudFound_P',
  'continuous_features': ['Age', 'Deductible'],
  'ordinal_features': ['WeekOfMonth', 'WeekOfMonthClaimed', 'DriverRating'],
  'categorical_features': ['Month',
   'DayOfWeek',
   'Make',
   'AccidentArea',
   'DayOfWeekClaimed',
   'MonthClaimed',
   'Sex',
   'MaritalStatus',
   'VehicleCategory',
   'VehiclePrice',
   'Days_Policy_Accident',
   'PastNumberOfClaims',
   'AgeOfVehicle',
   'AgeOfPolicyHolder

In [5]:
occurrence_config = (
    feature_manifest[
        "claim_occurrence"
    ]
)

frequency_config = (
    feature_manifest[
        "claim_frequency"
    ]
)

fraud_config = (
    feature_manifest[
        "fraud_detection"
    ]
)

In [7]:
print(
    "Occurrence IDs :",
    occurrence["IDpol"].nunique()
)

print(
    "Frequency IDs :",
    frequency["IDpol"].nunique()
)

Occurrence IDs : 676789
Frequency IDs : 676789


In [8]:
assert set(
    occurrence["IDpol"]
) == set(
    frequency["IDpol"]
)

print(
    "Les deux tâches utilisent les mêmes polices : OK"
)

Les deux tâches utilisent les mêmes polices : OK


In [13]:
fre_mtpl_split_base = (
    occurrence[
        [
            "IDpol",
            "has_claim"
        ]
    ]
    .copy()
)

dev_ids, test_ids = train_test_split(
    fre_mtpl_split_base["IDpol"],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=fre_mtpl_split_base["has_claim"]
)

dev_ids = set(
    dev_ids
)

test_ids = set(
    test_ids
)

occurrence_dev = (
    occurrence[
        occurrence["IDpol"]
        .isin(dev_ids)
    ]
    .copy()
)

occurrence_test = (
    occurrence[
        occurrence["IDpol"]
        .isin(test_ids)
    ]
    .copy()
)

frequency_dev = (
    frequency[
        frequency["IDpol"]
        .isin(dev_ids)
    ]
    .copy()
)

frequency_test = (
    frequency[
        frequency["IDpol"]
        .isin(test_ids)
    ]
    .copy()
)

In [14]:
assert (
    set(occurrence_dev["IDpol"])
    .isdisjoint(
        set(
            occurrence_test["IDpol"]
        )
    )
)

assert (
    set(frequency_dev["IDpol"])
    .isdisjoint(
        set(
            frequency_test["IDpol"]
        )
    )
)

print(
    "Aucun chevauchement Train/Test : OK"
)

Aucun chevauchement Train/Test : OK


In [15]:
print(
    "Occurrence DEV :",
    occurrence_dev.shape
)

print(
    "Occurrence TEST :",
    occurrence_test.shape
)

print()

print(
    "Frequency DEV :",
    frequency_dev.shape
)

print(
    "Frequency TEST :",
    frequency_test.shape
)

Occurrence DEV : (541431, 12)
Occurrence TEST : (135358, 12)

Frequency DEV : (541431, 12)
Frequency TEST : (135358, 12)


In [17]:
def target_distribution(
    df,
    target
):

    return (
        df[target]
        .value_counts(
            normalize=True
        )
        .sort_index()
        * 100
    )

print(
    "DEV"
)

print(
    target_distribution(
        occurrence_dev,
        "has_claim"
    )
)

print()

print(
    "TEST"
)

print(
    target_distribution(
        occurrence_test,
        "has_claim"
    )
)

DEV
has_claim
0    94.974983
1     5.025017
Name: proportion, dtype: float64

TEST
has_claim
0    94.974808
1     5.025192
Name: proportion, dtype: float64


In [20]:
occurrence_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)



In [21]:
occurrence_features = (
    occurrence_config[
        "numeric_features"
    ]
    +
    occurrence_config[
        "categorical_features"
    ]
)

In [22]:
X_occ_dev = (
    occurrence_dev[
        occurrence_features
    ].copy()
)

y_occ_dev = (
    occurrence_dev[
        occurrence_config[
            "target"
        ]
    ].copy()
)

In [23]:
X_occ_test = (
    occurrence_test[
        occurrence_features
    ].copy()
)

y_occ_test = (
    occurrence_test[
        occurrence_config[
            "target"
        ]
    ].copy()
)

In [24]:
occ_numeric = (
    occurrence_config[
        "numeric_features"
    ]
)

occ_categorical = (
    occurrence_config[
        "categorical_features"
    ]
)

print(
    "Numeric :",
    occ_numeric
)

print(
    "Categorical :",
    occ_categorical
)

Numeric : ['Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'log_density']
Categorical : ['Area', 'VehBrand', 'VehGas', 'Region']


In [25]:
numeric_linear_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [26]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [27]:
occurrence_linear_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_linear_pipeline,
                occ_numeric
            ),
            (
                "categorical",
                categorical_pipeline,
                occ_categorical
            )
        ]
    )
)

In [29]:
numeric_tree_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

occurrence_tree_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_tree_pipeline,
                occ_numeric
            ),
            (
                "categorical",
                categorical_pipeline,
                occ_categorical
            )
        ]
    )
)

## Métriques — Claim Occurrence

### Métrique principale

PR-AUC / Average Precision

Cette métrique est particulièrement adaptée aux problèmes
fortement déséquilibrés.

### Métriques secondaires

- ROC-AUC ;
- Precision ;
- Recall ;
- F1-score ;
- Brier Score ;
- matrice de confusion ;
- courbe de calibration.

Comme le modèle devra produire une probabilité de sinistre,
la calibration des probabilités est particulièrement importante.

In [33]:
frequency_features = (
    frequency_config[
        "numeric_features"
    ]
    +
    frequency_config[
        "categorical_features"
    ]
)

X_freq_dev = (
    frequency_dev[
        frequency_features
    ].copy()
)

y_freq_dev = (
    frequency_dev[
        frequency_config[
            "target"
        ]
    ].copy()
)

exposure_freq_dev = (
    frequency_dev[
        frequency_config[
            "exposure"
        ]
    ].copy()
)

X_freq_test = (
    frequency_test[
        frequency_features
    ].copy()
)

y_freq_test = (
    frequency_test[
        frequency_config[
            "target"
        ]
    ].copy()
)

exposure_freq_test = (
    frequency_test[
        frequency_config[
            "exposure"
        ]
    ].copy()
)

assert (
    X_freq_dev.shape[0]
    ==
    y_freq_dev.shape[0]
    ==
    exposure_freq_dev.shape[0]
)

assert (
    exposure_freq_dev > 0
).all()

print(
    "Frequency DEV : OK"
)

Frequency DEV : OK


In [35]:
frequency_strata = (
    frequency_dev[
        "ClaimNb"
    ] > 0
).astype(int)

In [36]:

frequency_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [37]:
observed_frequency = (
    y_freq_test.sum()
    /
    exposure_freq_test.sum()
)

In [38]:
fraud.groupby(
    "Year"
).agg(
    n=(
        "FraudFound_P",
        "size"
    ),
    frauds=(
        "FraudFound_P",
        "sum"
    ),
    fraud_rate=(
        "FraudFound_P",
        "mean"
    )
)

,n,frauds,fraud_rate
Year,,,
1994,6142,409,0.066591
1995,5195,301,0.057940
1996,4083,213,0.052168


In [39]:
fraud_train = (
    fraud[
        fraud["Year"] == 1994
    ]
    .copy()
)

fraud_val = (
    fraud[
        fraud["Year"] == 1995
    ]
    .copy()
)

fraud_test = (
    fraud[
        fraud["Year"] == 1996
    ]
    .copy()
)

In [40]:
for name, df in [
    ("TRAIN", fraud_train),
    ("VALIDATION", fraud_val),
    ("TEST", fraud_test)
]:

    print(
        name,
        df.shape
    )

    print(
        df["FraudFound_P"]
        .value_counts(
            normalize=True
        )
        .sort_index()
        * 100
    )

    print()

TRAIN (6142, 25)
FraudFound_P
0    93.340931
1     6.659069
Name: proportion, dtype: float64

VALIDATION (5195, 25)
FraudFound_P
0    94.205967
1     5.794033
Name: proportion, dtype: float64

TEST (4083, 25)
FraudFound_P
0    94.783248
1     5.216752
Name: proportion, dtype: float64



In [41]:
assert set(
    fraud_train["PolicyNumber"]
).isdisjoint(
    set(
        fraud_val["PolicyNumber"]
    )
)

assert set(
    fraud_train["PolicyNumber"]
).isdisjoint(
    set(
        fraud_test["PolicyNumber"]
    )
)

assert set(
    fraud_val["PolicyNumber"]
).isdisjoint(
    set(
        fraud_test["PolicyNumber"]
    )
)

print(
    "Fraud temporal split : OK"
)

Fraud temporal split : OK


In [45]:
fraud_features = (
    fraud_config[
        "continuous_features"
    ]
    +
    fraud_config[
        "ordinal_features"
    ]
    +
    fraud_config[
        "categorical_features"
    ]
)

X_fraud_train = (
    fraud_train[
        fraud_features
    ].copy()
)

y_fraud_train = (
    fraud_train[
        fraud_config[
            "target"
        ]
    ].copy()
)

X_fraud_val = (
    fraud_val[
        fraud_features
    ].copy()
)

y_fraud_val = (
    fraud_val[
        fraud_config[
            "target"
        ]
    ].copy()
)

X_fraud_test = (
    fraud_test[
        fraud_features
    ].copy()
)

y_fraud_test = (
    fraud_test[
        fraud_config[
            "target"
        ]
    ].copy()
)

In [46]:
assert "FraudFound_P" not in X_fraud_train.columns
assert "PolicyNumber" not in X_fraud_train.columns
assert "Year" not in X_fraud_train.columns

for feature in fraud_config[
    "potential_leakage_features"
]:

    assert feature not in X_fraud_train.columns

print(
    "Fraud leakage controls : OK"
)

Fraud leakage controls : OK


In [49]:
fraud_continuous = (
    fraud_config[
        "continuous_features"
    ]
)

fraud_ordinal = (
    fraud_config[
        "ordinal_features"
    ]
)

fraud_categorical = (
    fraud_config[
        "categorical_features"
    ]
)

In [50]:
fraud_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [51]:
fraud_ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [52]:
fraud_categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [53]:
fraud_linear_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "continuous",
                fraud_numeric_pipeline,
                fraud_continuous
            ),
            (
                "ordinal",
                fraud_ordinal_pipeline,
                fraud_ordinal
            ),
            (
                "categorical",
                fraud_categorical_pipeline,
                fraud_categorical
            )
        ]
    )
)

## Métriques — Fraud Detection

### Métrique principale

PR-AUC / Average Precision

La classe fraude est rare, ce qui rend PR-AUC plus informative
que l'accuracy seule.

### Métriques secondaires

- Recall fraude ;
- Precision fraude ;
- F1-score ;
- ROC-AUC ;
- matrice de confusion ;
- Brier Score si les probabilités sont exploitées.

Le seuil de classification sera choisi uniquement sur
le jeu de validation temporel 1995.

In [55]:
fre_mtpl_split = (
    occurrence[
        [
            "IDpol"
        ]
    ]
    .copy()
)

fre_mtpl_split["split"] = (
    np.where(
        fre_mtpl_split["IDpol"]
        .isin(test_ids),
        "test",
        "development"
    )
)

In [56]:
fre_mtpl_split[
    "split"
].value_counts()

split
development    541431
test           135358
Name: count, dtype: int64

In [57]:
fre_mtpl_split.to_csv(
    METADATA_DIR
    / "freMTPL2_split.csv",
    index=False
)

In [58]:
fraud_split = (
    fraud[
        [
            "PolicyNumber",
            "Year"
        ]
    ]
    .copy()
)

In [59]:
fraud_split["split"] = np.select(
    [
        fraud_split["Year"] == 1994,
        fraud_split["Year"] == 1995,
        fraud_split["Year"] == 1996
    ],
    [
        "train",
        "validation",
        "test"
    ],
    default="unknown"
)

In [60]:
fraud_split[
    "split"
].value_counts()

split
train         6142
validation    5195
test          4083
Name: count, dtype: int64

In [61]:
fraud_split.to_csv(
    METADATA_DIR
    / "fraud_temporal_split.csv",
    index=False
)

In [62]:
experimental_config = {

    "random_state":
        RANDOM_STATE,

    "claim_occurrence": {

        "test_size":
            0.20,

        "split":
            "stratified_random",

        "cv":
            "StratifiedKFold",

        "cv_folds":
            5,

        "primary_metric":
            "average_precision",

        "secondary_metrics": [
            "roc_auc",
            "precision",
            "recall",
            "f1",
            "brier_score"
        ]
    },

    "claim_frequency": {

        "test_size":
            0.20,

        "split":
            "same_as_claim_occurrence",

        "cv":
            "stratified_on_has_claim",

        "cv_folds":
            5,

        "primary_metric":
            "mean_poisson_deviance",

        "secondary_metrics": [
            "weighted_mae",
            "calibration"
        ]
    },

    "fraud_detection": {

        "split":
            "temporal",

        "train_year":
            1994,

        "validation_year":
            1995,

        "test_year":
            1996,

        "primary_metric":
            "average_precision",

        "secondary_metrics": [
            "recall",
            "precision",
            "f1",
            "roc_auc",
            "brier_score"
        ],

        "threshold_selection":
            "validation_only"
    }
}

In [63]:
with open(
    METADATA_DIR
    / "experimental_protocol.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experimental_config,
        f,
        indent=4,
        ensure_ascii=False
    )

In [64]:
print(
    json.dumps(
        experimental_config,
        indent=4,
        ensure_ascii=False
    )
)

{
    "random_state": 42,
    "claim_occurrence": {
        "test_size": 0.2,
        "split": "stratified_random",
        "cv": "StratifiedKFold",
        "cv_folds": 5,
        "primary_metric": "average_precision",
        "secondary_metrics": [
            "roc_auc",
            "precision",
            "recall",
            "f1",
            "brier_score"
        ]
    },
    "claim_frequency": {
        "test_size": 0.2,
        "split": "same_as_claim_occurrence",
        "cv": "stratified_on_has_claim",
        "cv_folds": 5,
        "primary_metric": "mean_poisson_deviance",
        "secondary_metrics": [
            "weighted_mae",
            "calibration"
        ]
    },
    "fraud_detection": {
        "split": "temporal",
        "train_year": 1994,
        "validation_year": 1995,
        "test_year": 1996,
        "primary_metric": "average_precision",
        "secondary_metrics": [
            "recall",
            "precision",
            "f1",
            "roc_auc"

# Test Set Policy

À partir de ce notebook :

## freMTPL2

Le jeu `test` de 20 % ne sera utilisé ni pour :

- le preprocessing ;
- le choix des modèles ;
- la sélection des features ;
- le tuning des hyperparamètres ;
- le choix du seuil ;
- la comparaison des expérimentations.

Il sera utilisé uniquement pour l'évaluation finale du modèle retenu.

## Fraud

L'année 1996 constitue le jeu de test final.

1994 est utilisé pour l'entraînement.

1995 est utilisé pour :

- comparer les modèles ;
- choisir les hyperparamètres ;
- choisir le seuil de classification.

1996 ne doit être consulté qu'après verrouillage de la configuration finale.

In [67]:
assert set(
    occurrence_dev["IDpol"]
).isdisjoint(
    set(
        occurrence_test["IDpol"]
    )
)
assert set(
    occurrence_dev["IDpol"]
).isdisjoint(
    set(
        occurrence_test["IDpol"]
    )
)
print(
    "Protocol integrity checks : OK"
)

Protocol integrity checks : OK


In [68]:
protocol_summary = pd.DataFrame([
    {
        "task": "Claim Occurrence",
        "problem": "Binary classification",
        "split": "80% DEV / 20% TEST stratified",
        "validation": "5-fold Stratified CV",
        "primary_metric": "PR-AUC"
    },

    {
        "task": "Claim Frequency",
        "problem": "Count / frequency",
        "split": "Same policies as Occurrence",
        "validation": "5-fold CV stratified on claim occurrence",
        "primary_metric": "Poisson Deviance"
    },

    {
        "task": "Fraud Detection",
        "problem": "Binary classification",
        "split": "1994 Train / 1995 Val / 1996 Test",
        "validation": "Temporal holdout",
        "primary_metric": "PR-AUC"
    }
])

protocol_summary

,task,problem,split,validation,primary_metric
0,Claim Occurrence,Binary classification,80% DEV / 20% TEST stratified,5-fold Stratified CV,PR-AUC
1,Claim Frequency,Count / frequency,Same policies as Occurrence,5-fold CV stratified on claim occurrence,Poisson Deviance
2,Fraud Detection,Binary classification,1994 Train / 1995 Val / 1996 Test,Temporal holdout,PR-AUC


In [70]:
protocol_summary.to_csv(
    REPORTS_DIR
    / "protocol_summary.csv",
    index=False
)

# Conclusion — Experimental Protocol

Le protocole expérimental est maintenant défini avant
tout entraînement de modèle.

## Claim Occurrence

- classification binaire ;
- split stratifié 80 % développement / 20 % test ;
- même test set conservé pour tous les modèles ;
- validation croisée stratifiée sur le développement ;
- PR-AUC comme métrique principale ;
- calibration également étudiée.

## Claim Frequency

- cible de comptage `ClaimNb` ;
- `Exposure` conservée séparément ;
- même partition de polices que Claim Occurrence ;
- Poisson Deviance comme métrique principale ;
- calibration fréquence observée/prédite étudiée.

## Fraud Detection

- classification binaire déséquilibrée ;
- protocole temporel :
  - 1994 : train ;
  - 1995 : validation ;
  - 1996 : test ;
- PR-AUC comme métrique principale ;
- seuil de classification choisi uniquement sur validation.

## Protection contre le Data Leakage

Toutes les transformations apprises seront placées dans des pipelines.

Les imputations, encodages et standardisations seront ajustés
uniquement sur les données d'entraînement.

Le jeu de test final ne participera ni à la sélection du modèle,
ni au tuning, ni au choix du seuil.

## Étape suivante

`06_claim_occurrence_models.ipynb`

Nous commencerons par établir des baselines reproductibles avant
de tester des modèles plus complexes.